In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima 
import joblib


In [ ]:


def create_fourier_terms(data, daily_k=6, weekly_k=3):
    X = pd.DataFrame(index=data.index)

    half_hour_slot = data["datetime_local"].dt.hour * 2 + data["datetime_local"].dt.minute // 30

    week_slot = data["datetime_local"].dt.dayofweek * 48 + half_hour_slot

    #daily Fourier terms
    for k in range(1, daily_k + 1):

        X[f"daily_sin_{k}"] = np.sin(2 * np.pi * k * half_hour_slot / 48)
        X[f"daily_cos_{k}"] = np.cos(2 * np.pi * k * half_hour_slot / 48)


    #weekly Fourier terms
    for k in range(1, weekly_k + 1):

        X[f"weekly_sin_{k}"] = np.sin(2 * np.pi * k * week_slot / 336)
        X[f"weekly_cos_{k}"] = np.cos(2 * np.pi * k * week_slot / 336)

    return X

In [ ]:
fourier = create_fourier_terms(df, daily_k=6, weekly_k=3)
print(fourier.shape)
print(fourier.head())

In [ ]:
y_train = train["ND"].copy()
y_validation = validation["ND"].copy()
y_test = test["ND"].copy()

X_train = fourier.loc[train.index].copy()
X_validation = fourier.loc[validation.index].copy()
X_test = fourier.loc[test.index].copy()

In [ ]:
X_train.shape, X_validation.shape, X_test.shape

In [ ]:
#create order-search sample for 2023
order_search_y = y_train.loc["2023-01-01":"2023-12-31"]

order_search_X = X_train.loc[order_search_y.index]

print("Target shape:", order_search_y.shape)
print("Fourier shape:", order_search_X.shape)
print("Indexes match:",order_search_y.index.equals(order_search_X.index))

In [ ]:
#search for ARIMA error order


arima_results = auto_arima(
    order_search_y, X=order_search_X,
    seasonal=False, d=0,
    start_p=2, start_q=1,
    max_p=6,max_q=5,
    information_criterion="aic",
    stepwise=True,trace=True,
    error_action="ignore",
    suppress_warnings=True,
    with_intercept=True
)

In [ ]:
arima_results.summary()

In [ ]:
arima_results.plot_diagnostics(figsize=(12, 8))
plt.show()

In [ ]:
residuals = arima_results.resid()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_acf(residuals,lags=400,ax=ax)
ax.axvline(48, linestyle="--", alpha=0.6)
ax.axvline(96, linestyle="--", alpha=0.6)
ax.axvline(336, linestyle="--", alpha=0.6)

plt.show()

In [ ]:
y_pred =arima_results.predict(n_periods=len(validation), X=X_validation)
mae = mean_absolute_error(y_validation, y_pred)
mae

In [ ]:

#create lags of 24h and weekly
df['lag_72h'] = df['ND'].shift(144)
df['lag_48h'] = df['ND'].shift(96)
df['lag_168h'] = df['ND'].shift(336)

#rolling features 
#df['roll_mean_24h'] = df['ND'].shift(1).rolling(window = 48).mean()
#df['roll_mean_168h'] = df['ND'].shift(1).rolling(window = 336).mean()
#df['roll_std_24h'] = df['ND'].shift(1).rolling(window = 48).std()
#df['roll_std_168h'] = df['ND'].shift(1).rolling(window = 336).std()

In [ ]:
lag_cols = ['lag_72h', 'lag_48h', 'lag_168h']
X = pd.concat([fourier,df[lag_cols]],axis=1)

X_train = X.loc[train.index].copy()
X_validation = X.loc[validation.index].copy()
X_test = X.loc[test.index].copy()

order_search_y = y_train.loc["2023-01-01":"2023-12-31"]
order_search_X2 = X_train.loc[order_search_y.index]

In [ ]:
arima_results_2 = auto_arima(
    order_search_y, X=order_search_X2,
    seasonal=False, d=0,
    start_p=2, start_q=1,
    max_p=6,max_q=5,
    information_criterion="aic",
    stepwise=True,trace=True,
    error_action="ignore",
    suppress_warnings=True,
    with_intercept=True
)

In [ ]:
arima_results_2.summary()

In [ ]:
arima_results_2.plot_diagnostics(figsize=(12, 8))
plt.show()

In [ ]:
residuals2 = arima_results_2.resid()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_acf(residuals,lags=400,ax=ax)
ax.axvline(48, linestyle="--", alpha=0.6)
ax.axvline(96, linestyle="--", alpha=0.6)
ax.axvline(336, linestyle="--", alpha=0.6)

plt.show()

In [ ]:
y_pred_1 =arima_results_2.predict(n_periods=len(validation), X=X_validation)
mae = mean_absolute_error(y_validation, y_pred_1)
mae

In [ ]:

model_seasonal = SARIMAX(
    order_search_y,
    exog=order_search_X2,
    order=(2, 0, 1),
    seasonal_order=(1, 0, 0, 48),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False
)

results_seasonal = model_seasonal.fit()

In [ ]:
results_seasonal.summary()

In [ ]:
results_seasonal.plot_diagnostics(figsize=(12, 8))
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error


In [ ]:
y_pred_2 =(results_seasonal.get_prediction(start=validation.index[0], 
                                         end=validation.index[-1], exog=X_validation))
mae = mean_absolute_error(y_validation, y_pred_2.predicted_mean)
mae